# 토이 파인튜닝: LoRA-A(정보 추출) / LoRA-B(응답 생성)

**목표**: 받은 100건으로 작은 모델에 LoRA를 붙여 실제로 학습이 되는지 확인하고, vLLM에서 A·B 어댑터를 골라 호출하는 시간까지 재 본다.

- 런타임: Colab **T4 GPU** 이상 (런타임 → 런타임 유형 변경 → T4 GPU)
- 모델: `Qwen3-1.7B` (Apache 2.0, 작아서 T4에서 빠름). 한국어 모델로 바꾸려면 `MODEL`만 수정
- 데이터: `data_slim` (프롬프트를 약 1/10로 줄인 버전). 원본 그대로는 `data_full`
- 순서: ① 설치 → ② 데이터 업로드 → ③ 설정 → ③-1 평가 함수 → ③-2 학습 전 점수(선택) → ④ 학습 → ⑤ 평가 → ⑥ 저장
  → `TASK`를 `"B"`로 바꾸고 **런타임 다시 시작 후** ③부터 다시 → ⑦ vLLM 시간 측정(새 런타임)

## ① 설치

In [ ]:
!pip install -q unsloth

## ② 데이터 업로드 — `toy_finetune.zip`을 올리고 압축 해제

In [ ]:
from google.colab import files
up = files.upload()            # toy_finetune.zip 선택
!unzip -o -q toy_finetune.zip
!ls data_slim

## ③ 설정 — A 학습 후 B 학습할 때 여기 `TASK`만 바꾸기

In [ ]:
TASK     = "A"                    # "A" = 정보 추출(JSON),  "B" = 응답 문장
DATA_DIR = "data_slim"            # 원본 길이로 해보려면 "data_full" (MAX_LEN 12288 필요, 훨씬 느림)
MODEL    = "unsloth/Qwen3-1.7B"   # 예: "kakaocorp/kanana-1.5-2.1b-instruct-2505" 로 교체 가능(아래 템플릿 주의)
MAX_LEN  = 4096
EPOCHS   = 3
OUT_DIR  = f"lora_{TASK}"

## ③-1 평가 함수 (학습 전·후 공통)

In [ ]:
import json, time, re, torch

def generate(model, tokenizer, msgs, max_new):
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    t = time.time()
    out = model.generate(**ids, max_new_tokens=max_new, do_sample=False)
    text = tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    text = re.sub(r"^\s*<think>.*?</think>", "", text, flags=re.S)   # 빈 생각 블록이 붙어 나오면 제거
    return text.strip(), time.time()-t

def parse_json(text):
    """```json ... ``` 같은 코드블록 표시는 벗겨 내고 읽기 (학습 전·후 똑같이 적용)"""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    d = json.loads(text)
    if not isinstance(d, dict):
        raise ValueError("JSON 객체가 아님")
    return d

def fact_keys(d):
    """비교 단위: (문항, 필드, 값, 상태) + 의도 + 관계. 키가 빠진 항목도 에러 없이 처리"""
    as_list = lambda x: x if isinstance(x, list) else []
    ok = lambda x: isinstance(x, dict)
    return {(f.get("item_id"), f.get("field"), json.dumps(f.get("value"), ensure_ascii=False), f.get("semantic_status"))
            for f in as_list(d.get("facts")) if ok(f)} \
         | {(i.get("item_id"), "INTENT", i.get("intent")) for i in as_list(d.get("intents")) if ok(i)} \
         | {(r.get("item_id"), "REL", r.get("relation")) for r in as_list(d.get("relations")) if ok(r)}

def evaluate(model, tokenizer, task, data_dir, label="", show=True):
    """eval 13건 평가. A는 지표 dict, B는 응답 목록을 돌려준다"""
    rows = [json.loads(l) for l in open(f"{data_dir}/{task}_eval.jsonl", encoding="utf-8")]
    ok_json = exact = 0; tp = fp = fn = 0; ev_ok = ev_pos = ev_all = 0; secs = []; outs = []
    for r in rows:
        msgs, gold = r["messages"][:-1], r["messages"][-1]["content"]
        pred, sec = generate(model, tokenizer, msgs, 512 if task == "A" else 128)
        secs.append(sec); outs.append({"id": r["id"], "gold": gold, "pred": pred})
        if show:
            print(f"\n[{label} {r['id']} | {r['category']} | {sec:.2f}s]")
            print("사용자:", json.loads(msgs[1]["content"])["context"][-1]["content"])
            print("정답  :", gold); print("예측  :", pred)
        if task != "A":
            continue
        G = fact_keys(json.loads(gold))
        try:
            pd = parse_json(pred); P = fact_keys(pd); ok_json += 1
        except Exception:
            fn += len(G); continue
        tp += len(P & G); fp += len(P - G); fn += len(G - P); exact += (P == G)
        # 근거 검사: evidence.turn_index가 가리키는 사용자 발화에서 확인
        #   ev_ok  = 근거 문자열이 그 발화 안에 들어 있음 (위치 숫자는 틀려도 됨)
        #   ev_pos = start/end로 잘라낸 결과까지 정확히 일치
        turns = {t["turn_index"]: t["content"] for t in json.loads(msgs[1]["content"])["context"] if t["role"] == "user"}
        for f in pd["facts"] if isinstance(pd.get("facts"), list) else []:
            e = f.get("evidence") if isinstance(f, dict) else None
            if not isinstance(e, dict): e = {}
            src = turns.get(e.get("turn_index"), "")
            txt = e.get("text") if isinstance(e.get("text"), str) else ""
            ev_all += 1
            ev_ok  += bool(txt) and txt in src
            ev_pos += isinstance(e.get("start"), int) and isinstance(e.get("end"), int) and bool(txt) and src[e["start"]:e["end"]] == txt
    avg = round(sum(secs) / len(secs), 2)
    if task != "A":
        print(f"\n[{label}] 평균 생성 시간 {avg}s (HF generate 기준, vLLM보다 느림)")
        return {"sec": avg, "outputs": outs}
    n = len(rows); prec = tp/max(tp+fp,1); rec = tp/max(tp+fn,1); f1 = 2*prec*rec/max(prec+rec,1e-9)
    m = {"json_ok": f"{ok_json}/{n}", "exact": f"{exact}/{n}", "P": round(prec,2), "R": round(rec,2), "F1": round(f1,2),
         "evidence_text": f"{ev_ok}/{ev_all}", "evidence_pos": f"{ev_pos}/{ev_all}", "sec": avg, "outputs": outs}
    print(f"\n[{label}] JSON 파싱 성공 {m['json_ok']} | 완전일치 {m['exact']} | 항목 P {prec:.2f} R {rec:.2f} F1 {f1:.2f}"
          f" | 근거 문자열 유효 {m['evidence_text']} | 근거 위치 정확 {m['evidence_pos']} | 평균 {avg}s")
    return m


## ③-2 (선택) 학습 전 기준 점수
원본 모델(어댑터 없음)로 같은 13건을 평가 → `baseline_{TASK}.json` 저장.
발표에서 "파인튜닝 전 → 후" 비교에 사용. 원본은 형식을 잘 못 맞추는 게 정상이라 점수가 낮아도 문제 아님.

In [ ]:
# 파인튜닝 전 원본 모델 점수 — 학습 효과 비교용 기준 (TASK마다 한 번)
# 시간이 없으면 건너뛰어도 학습에는 영향 없음
from unsloth import FastLanguageModel
base_model, base_tok = FastLanguageModel.from_pretrained(model_name=MODEL, max_seq_length=MAX_LEN, load_in_4bit=False)
FastLanguageModel.for_inference(base_model)
base_metrics = evaluate(base_model, base_tok, TASK, DATA_DIR, label="학습 전")
json.dump(base_metrics, open(f"baseline_{TASK}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

import gc
del base_model; gc.collect(); torch.cuda.empty_cache()   # 메모리 비우고 ④ 학습으로


## ④ 모델 불러오기 + LoRA 붙이기 + 학습

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_LEN, load_in_4bit=False)   # 1.7B는 16bit로도 T4에 들어감

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

ds = load_dataset("json", data_files={"train": f"{DATA_DIR}/{TASK}_train.jsonl",
                                      "eval":  f"{DATA_DIR}/{TASK}_eval.jsonl"})

def to_text(ex):
    # Qwen3: 생각 모드 끄고(enable_thinking=False) 대화 전체를 학습용 문자열로
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, enable_thinking=False)}
ds = ds.map(to_text)

lens = [len(tokenizer(t).input_ids) for t in ds["train"]["text"]]
print(f"토큰 길이: 평균 {sum(lens)//len(lens)}, 최대 {max(lens)}  (MAX_LEN={MAX_LEN})")
assert max(lens) <= MAX_LEN, "MAX_LEN을 늘리세요"

# TRL 버전에 따라 인자 이름이 다름 → 새 이름으로 먼저 시도하고, 안 되면 옛 이름으로
cfg_kw = dict(dataset_text_field="text",
              per_device_train_batch_size=2, gradient_accumulation_steps=4,
              num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=3,
              lr_scheduler_type="linear", logging_steps=2, optim="adamw_8bit",
              seed=42, output_dir="outputs", report_to="none")
try:
    cfg = SFTConfig(max_length=MAX_LEN, **cfg_kw)        # 최신 TRL
except TypeError:
    cfg = SFTConfig(max_seq_length=MAX_LEN, **cfg_kw)    # 옛 TRL
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds["train"], args=cfg)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds["train"], args=cfg)

# 정답(assistant) 부분만 손실 계산 — 질문/설문 카탈로그는 외우지 않게
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

stats = trainer.train()
print(stats)

## ⑤ 학습 후 평가 — 떼어 둔 13건 (같은 템플릿 데이터라 '형식을 배웠는지' 확인용)
결과는 `finetuned_{TASK}.json`에 저장. ③-2를 돌렸다면 학습 전과 나란히 비교해 줍니다.

In [ ]:
FastLanguageModel.for_inference(model)
ft_metrics = evaluate(model, tokenizer, TASK, DATA_DIR, label="학습 후")
json.dump(ft_metrics, open(f"finetuned_{TASK}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

# ③-2를 돌렸다면 학습 전과 나란히 비교
if "base_metrics" in globals() and TASK == "A":
    print("\n지표            학습 전      학습 후")
    for k, name in [("json_ok","JSON 파싱"), ("exact","완전일치"), ("P","Precision"), ("R","Recall"), ("F1","F1"),
                    ("evidence_text","근거 문자열"), ("evidence_pos","근거 위치"), ("sec","평균 시간(s)")]:
        print(f"{name:<14}{str(base_metrics[k]):>10}{str(ft_metrics[k]):>12}")
elif "base_metrics" in globals():
    print("\n학습 전 / 후 응답 비교")
    for b, f in zip(base_metrics["outputs"], ft_metrics["outputs"]):
        print(f"\n[{f['id']}]\n  정답   : {f['gold']}\n  학습 전: {b['pred']}\n  학습 후: {f['pred']}")


## ⑥ 어댑터 저장 + 다운로드 (병합·GGUF 변환 안 함 → vLLM에 그대로 붙임)

In [ ]:
model.save_pretrained(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
!zip -q -r {OUT_DIR}.zip {OUT_DIR}
files.download(f"{OUT_DIR}.zip")
import os
for f in (f"baseline_{TASK}.json", f"finetuned_{TASK}.json"):
    if os.path.exists(f): files.download(f)   # 지표 파일도 받아 두기 (결과 요약에 사용)
# → 이제 런타임 다시 시작, ③에서 TASK="B"로 바꿔 한 번 더

## ⑦ vLLM에서 A·B 어댑터 골라 호출 + 시간 측정 (명찬 파트)
**새 런타임**에서 실행. `lora_A.zip`, `lora_B.zip`, `toy_finetune.zip`을 올린 뒤 진행.
노트북 안에서 `LLM(...)`으로 띄우면 Colab 출력창 때문에 `io.UnsupportedOperation: fileno` 에러가 나므로,
4090 서버와 같은 방식으로 `vllm serve` 서버를 띄우고 `bench.py`로 잰다.
T4는 4090보다 느리므로 절대 시간보다 **A·B·원본 비율, 어댑터 유무 차이**를 보는 용도.

In [ ]:
!pip install -q vllm
from google.colab import files
files.upload()   # lora_A.zip, lora_B.zip, toy_finetune.zip
!unzip -o -q lora_A.zip && unzip -o -q lora_B.zip && unzip -o -q toy_finetune.zip

In [ ]:
# 서버를 백그라운드로 실행 (로그는 vllm.log)
!nohup vllm serve Qwen/Qwen3-1.7B --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --enable-prefix-caching --enable-lora --max-lora-rank 16 --max-loras 2 --lora-modules lora_A=/content/lora_A lora_B=/content/lora_B --port 8000 > vllm.log 2>&1 &


In [ ]:
# 준비될 때까지 대기 (모델 다운로드·컴파일 포함, 최대 10분)
import time, urllib.request
for i in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=3)
        print(f"준비 완료 ({i*5}초)"); break
    except Exception:
        time.sleep(5)
else:
    print("10분 안에 안 떴음 → 아래 로그 확인")
!tail -n 15 vllm.log


In [ ]:
# A·B·원본 시간 측정 (예열 1회 제외, 13건 × 3회) → 결과 파일 다운로드
!python bench.py --save bench_t4.json
from google.colab import files; files.download("bench_t4.json")
